<a href="https://colab.research.google.com/github/warry258/colab/blob/main/%E6%89%B9%E9%87%8F%E6%8F%90%E5%8F%96%E6%8E%A7%E5%88%B6%E5%9B%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 挂载完成！")

In [ ]:
# @title 2. 环境配置与下载预处理模型
import os
import sys

# 1. 安装核心依赖（仅需轻量级底层库，抛弃臃肿的框架）
!pip install -q controlnet_aux==0.0.9 opencv-python-headless pillow accelerate

import torch
import cv2
import numpy as np
from PIL import Image
from controlnet_aux import (
    CannyDetector,
    OpenposeDetector,
    HEDdetector,
    MLSDdetector,
    MidasDetector,
    PidiNetDetector
)

# 2. 环境检测
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ 当前运算环境: 【{device.upper()}】")

# 设置模型缓存目录
os.environ["HF_HOME"] = "/content/ckpts"
os.makedirs("/content/ckpts", exist_ok=True)
os.makedirs("/content/input_images", exist_ok=True)
os.makedirs("/content/output_controls", exist_ok=True)

# 3. 预加载所有核心模型字典
preprocessors = {}

def load_models():
    print("⏳ 正在拉取并加载预处理器模型 (初次执行需下载少量文件，请稍候)...")
    try:
        preprocessors["canny"]   = CannyDetector()
        preprocessors["pose"]    = OpenposeDetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["depth"]   = MidasDetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["mlsd"]    = MLSDdetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["hed"]     = HEDdetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["pidinet"] = PidiNetDetector.from_pretrained("lllyasviel/Annotators").to(device)
        print("🎉 所有模型加载就绪！")
    except Exception as e:
        print(f"❌ 模型加载发生异常: {e}")

load_models()

In [ ]:
# @title 3. ControlNet 提取器
import os, gc, time, traceback, torch, io, ipywidgets as widgets
from PIL import Image
from IPython.display import display, Image as IPImage
from google.colab import files as colab_files

# ══════════════ 全局状态 ══════════════
CONFIG = {
    "image_path": "", "extractor": "canny", "resolution": 512,
    "canny_low": 100, "canny_high": 200,
    "pose_hand": False, "pose_face": False,
    "mlsd_thr_v": 0.1, "mlsd_thr_d": 0.1, "scribble_algo": "hed",
}

# ══════════════ 组件 ══════════════
w_source = widgets.ToggleButtons(
    options=["📁 填写路径", "⬆️ 上传文件"],
    value="📁 填写路径", description="图片来源:")

w_path = widgets.Text(
    value="", placeholder="/content/drive/MyDrive/your_image.png",
    description="图片路径:", layout=widgets.Layout(width="500px"),
    style={"description_width": "70px"})

btn_upload = widgets.Button(description="点击上传图片", button_style="primary",
    layout=widgets.Layout(width="160px", display="none"))
w_upload_label = widgets.Label(value="", layout=widgets.Layout(display="none"))

w_extractor = widgets.ToggleButtons(
    options=[("Canny 线稿","canny"),("Depth 深度","depth"),
             ("Pose 姿态","pose"),("MLSD 直线","mlsd"),
             ("HED 软边缘","hed"),("Scribble 涂鸦","scribble")],
    value="canny", description="提取器:")

w_res = widgets.SelectionSlider(
    options=[256,320,384,448,512,576,640,768,896,1024],
    value=512, description="分辨率:", style={"description_width":"50px"})

# 专属参数
w_cl = widgets.IntSlider(value=100, min=0, max=255, step=5, description="低阈值:", style={"description_width":"50px"})
w_ch = widgets.IntSlider(value=200, min=0, max=255, step=5, description="高阈值:", style={"description_width":"50px"})
box_canny = widgets.VBox([widgets.HTML("<b>Canny 参数</b>"), widgets.HBox([w_cl, w_ch])],
    layout=widgets.Layout(border="1px solid #ddd", padding="8px", margin="4px 0"))

w_ph = widgets.Checkbox(value=False, description="检测手部", indent=False)
w_pf = widgets.Checkbox(value=False, description="检测面部", indent=False)
box_pose = widgets.VBox([widgets.HTML("<b>Pose 参数</b>"), widgets.HBox([w_ph, w_pf])],
    layout=widgets.Layout(border="1px solid #ddd", padding="8px", margin="4px 0"))

w_mv = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description="得分阈值:", style={"description_width":"60px"}, readout_format=".2f")
w_md = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description="距离阈值:", style={"description_width":"60px"}, readout_format=".2f")
box_mlsd = widgets.VBox([widgets.HTML("<b>MLSD 参数</b>"), widgets.HBox([w_mv, w_md])],
    layout=widgets.Layout(border="1px solid #ddd", padding="8px", margin="4px 0"))

w_sc = widgets.ToggleButtons(options=["hed","pidinet"], value="hed", description="底层算法:")
box_scribble = widgets.VBox([widgets.HTML("<b>Scribble 参数</b>"), w_sc],
    layout=widgets.Layout(border="1px solid #ddd", padding="8px", margin="4px 0"))

btn_run = widgets.Button(description="🚀 开始提取", button_style="success",
    layout=widgets.Layout(width="180px", height="40px", margin="10px 0 0 0"))
btn_clear = widgets.Button(description="🗑️ 清空记录", button_style="warning",
    layout=widgets.Layout(width="120px", height="40px", margin="10px 0 0 6px"))

# ══════════════ Output 管理（新建实例彻底清除缓存）══════════════
def make_out():
    return widgets.Output(layout=widgets.Layout(
        border="1px solid #ccc", padding="10px",
        margin="8px 0", min_height="50px"))

out_log = make_out()
out_container = widgets.VBox([out_log])

# ══════════════ observe → CONFIG ══════════════
def on_source(c):
    use_up = c["new"] == "⬆️ 上传文件"
    w_path.layout.display         = "none" if use_up else "flex"
    btn_upload.layout.display     = "flex" if use_up else "none"
    w_upload_label.layout.display = "flex" if use_up else "none"

def on_extractor(c):
    CONFIG["extractor"] = c["new"]
    box_canny.layout.display    = "flex" if c["new"] == "canny"    else "none"
    box_pose.layout.display     = "flex" if c["new"] == "pose"     else "none"
    box_mlsd.layout.display     = "flex" if c["new"] == "mlsd"     else "none"
    box_scribble.layout.display = "flex" if c["new"] == "scribble" else "none"

w_source.observe(on_source, names="value")
w_path.observe(lambda c: CONFIG.update({"image_path": c["new"].strip()}), names="value")
w_extractor.observe(on_extractor, names="value")
w_res.observe(lambda c: CONFIG.update({"resolution": c["new"]}), names="value")
w_cl.observe(lambda c: CONFIG.update({"canny_low":     c["new"]}), names="value")
w_ch.observe(lambda c: CONFIG.update({"canny_high":    c["new"]}), names="value")
w_ph.observe(lambda c: CONFIG.update({"pose_hand":     c["new"]}), names="value")
w_pf.observe(lambda c: CONFIG.update({"pose_face":     c["new"]}), names="value")
w_mv.observe(lambda c: CONFIG.update({"mlsd_thr_v":    c["new"]}), names="value")
w_md.observe(lambda c: CONFIG.update({"mlsd_thr_d":    c["new"]}), names="value")
w_sc.observe(lambda c: CONFIG.update({"scribble_algo": c["new"]}), names="value")

# ══════════════ 上传 ══════════════
def do_upload(b):
    uploaded = colab_files.upload()
    if not uploaded:
        w_upload_label.value = "❌ 未选择文件"
        return
    fname = list(uploaded.keys())[0]
    os.makedirs("/content/input_images", exist_ok=True)
    save = f"/content/input_images/{fname}"
    with open(save, "wb") as f:
        f.write(uploaded[fname])
    CONFIG["image_path"] = save
    w_upload_label.value = f"✅ {fname}"

btn_upload.on_click(do_upload)

# ══════════════ 提取 ══════════════
def do_extract(b):
    global out_log
    out_log = make_out()
    out_container.children = [out_log]      # 先不放按钮，提取完再加

    path = CONFIG["image_path"]
    if not path or not os.path.exists(path):
        out_log.append_stdout(f"❌ 找不到图片：{path or '未设置'}\n")
        return

    ext = CONFIG["extractor"]
    res = CONFIG["resolution"]
    out_log.append_stdout(f"⏳ 提取中：{ext}  分辨率={res} ...\n")

    out_img = None
    try:
        img = Image.open(path).convert("RGB")
        out_log.append_stdout(f"🖼️ 图片尺寸：{img.size}\n")

        with torch.inference_mode():
            if ext == "canny":
                out_img = preprocessors["canny"](img,
                    low_threshold=CONFIG["canny_low"],
                    high_threshold=CONFIG["canny_high"],
                    image_resolution=res)
            elif ext == "depth":
                out_img = preprocessors["depth"](img, image_resolution=res)
            elif ext == "pose":
                out_img = preprocessors["pose"](img, include_body=True,
                    include_hand=CONFIG["pose_hand"],
                    include_face=CONFIG["pose_face"],
                    image_resolution=res)
            elif ext == "mlsd":
                out_img = preprocessors["mlsd"](img,
                    thr_v=CONFIG["mlsd_thr_v"],
                    thr_d=CONFIG["mlsd_thr_d"],
                    image_resolution=res)
            elif ext == "hed":
                out_img = preprocessors["hed"](img, image_resolution=res)
            elif ext == "scribble":
                algo = preprocessors["hed"] if CONFIG["scribble_algo"] == "hed" \
                       else preprocessors["pidinet"]
                out_img = algo(img, scribble=True, image_resolution=res)

        os.makedirs("/content/output_controls", exist_ok=True)
        save_path = f"/content/output_controls/result_{ext}_{int(time.time())}.png"
        out_img.save(save_path)
        out_log.append_stdout(f"🎉 已保存：{save_path}\n")

        # 预览缩略图
        thumb = out_img.copy()
        thumb.thumbnail((600, 600))
        buf = io.BytesIO()
        thumb.save(buf, format="PNG")
        out_log.append_display_data(IPImage(data=buf.getvalue()))
        out_log.append_stdout(f"☝️ 上图为预览缩略图，点击下方按钮下载原图（{out_img.size[0]}x{out_img.size[1]}）\n")

        # 下载原图按钮（放在 out_log 外面，避免触发重渲染）
        btn_dl = widgets.Button(description=f"⬇️ 下载原图 ({out_img.size[0]}x{out_img.size[1]})", button_style="info",
            layout=widgets.Layout(width="240px", margin="4px 0"))
        btn_dl.on_click(lambda b, p=save_path: colab_files.download(p))
        out_container.children = [out_log, btn_dl]

    except Exception:
        out_log.append_stdout(traceback.format_exc())
    finally:
        del img
        if out_img is not None: del out_img
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def do_clear(b):
    global out_log
    out_log = make_out()
    out_container.children = [out_log]

btn_run.on_click(do_extract)
btn_clear.on_click(do_clear)

# ══════════════ 初始化 ══════════════
on_source({"new": w_source.value})
on_extractor({"new": w_extractor.value})

# ══════════════ 布局 ══════════════
display(widgets.VBox([
    widgets.HTML("<h2 style='margin:0 0 8px'>🎯 ControlNet 提取器</h2>"),
    widgets.HTML("<b>① 图片来源</b>"),
    w_source, w_path, widgets.HBox([btn_upload, w_upload_label]),
    widgets.HTML("<b>② 提取器</b>"),
    w_extractor,
    widgets.HTML("<b>③ 分辨率</b>"),
    widgets.HBox([w_res, widgets.Label("💡 CPU 推荐 512 以下")]),
    widgets.HTML("<b>④ 专属参数</b>"),
    box_canny, box_pose, box_mlsd, box_scribble,
    widgets.HTML("<hr style='margin:8px 0'>"),
    widgets.HBox([btn_run, btn_clear]),
    out_container,
], layout=widgets.Layout(
    padding="16px", border="2px solid #c7d2fe",
    border_radius="10px", width="720px")))

In [ ]:
# @title 4. 一键导出提取结果 (保存至网盘或下载到本地)
import os
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

w_dest_path = widgets.Text(value="/content/drive/MyDrive/ControlNet_提取结果", description="云盘目标目录", style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
btn_copy_drive = widgets.Button(description="📂 拷贝至 Google Drive", button_style="success", layout=widgets.Layout(width='180px'))
btn_download_zip = widgets.Button(description="📦 打包下载到本地", button_style="info", layout=widgets.Layout(width='180px'))
out_export = widgets.Output()

out_dir = "/content/output_controls"

def copy_to_drive(_):
    with out_export:
        clear_output()
        if not os.path.exists(out_dir) or not os.listdir(out_dir):
            print("⚠️ 暂无提取结果，请先在上方执行提取操作。")
            return
        dest = w_dest_path.value.strip()
        os.makedirs(dest, exist_ok=True)
        !cp -r {out_dir}/* "{dest}/"
        print(f"✅ 已成功拷贝 {len(os.listdir(out_dir))} 个文件至：{dest}")

def download_local(_):
    with out_export:
        clear_output()
        if not os.path.exists(out_dir) or not os.listdir(out_dir):
            print("⚠️ 暂无提取结果，请先在上方执行提取操作。")
            return
        print("⏳ 正在打包 zip 文件，请稍候...")
        shutil.make_archive("/content/Extracted_ControlNet", 'zip', out_dir)
        print("✅ 打包完成！浏览器即将开始下载...")
        files.download("/content/Extracted_ControlNet.zip")

btn_copy_drive.on_click(copy_to_drive)
btn_download_zip.on_click(download_local)

ui_export = widgets.VBox([
    widgets.HTML("<h3>💾 导出与下载</h3>"),
    widgets.HBox([w_dest_path, btn_copy_drive]),
    widgets.HTML("<br><i>或者直接下载到本地电脑：</i>"),
    btn_download_zip,
    out_export
])
display(ui_export)